In [29]:
from pathlib import Path ## to use the Path class for file handling
from google import genai ## to use the Gemini API client
from datetime import datetime ## to get the current date and time
import json ## to handle JSON data
import pandas as pd ## to handle data in DataFrame format
import os ## to handle environment variables
import dtale

In [ ]:
# Get current local date and time
now = datetime.now()

##extract the date in YYYY-MM-DD format
today = now.strftime("%Y-%m-%d")
print("\nToday's date: " + today)


2026-07-17 15:59:33.798969

Today's date: 2026-07-17


In [ ]:
#configure the paths for the notes and prompt template files
notes_path = Path("data/import/sample_notes.md")
prompt_path = Path("data/import/extraction_prompt_template.md")

In [ ]:
#storing the contents of the notes and prompt template files in variables
raw_notes = notes_path.read_text(encoding="utf-8")
prompt_template = prompt_path.read_text(encoding="utf-8")

print("Notes:")
print(raw_notes)

print("\nPrompt template:")
print(prompt_template)

Notes:
- ACE promotional layout meet 16th aug-6pm    
- ML pipeline debug 17th aug- YOLO- 8pm
- calisthenics routine this sunday
- shoot YouTube banter vid (this weekend)
- ask papa about schedule (today or tmmrw)
- flute practice 30 mins
- finish CCUS report by tomorrow please!!!!

Prompt template:
# Notes Extraction Prompt (Phase 2)

Use this as the system/instruction prompt when calling the LLM to parse raw notes
into structured tasks. Inject `{today_date}`, `{timezone}`, and `{raw_notes}` at
call time.

---

## Prompt Template

```text
You are a task extraction engine for a personal scheduling system.

Today's date is: {today_date}
Timezone is: {timezone}

You will be given raw, unstructured notes - one item per line, written quickly
and informally. Extract each line into a structured JSON object.

For each item, output exactly these fields:
- "title": short, cleaned-up name of the task/event (string)
- "event_type": either "fixed_time" (a meeting, call, appointment, or suggested
 

In [ ]:
#configure the path for the Gemini API key file

gemini_api_key_path = Path("data/API_tokens_values/gemini_api_key.txt")

if not gemini_api_key_path.exists():
    raise FileNotFoundError(f"Gemini API key file not found: {gemini_api_key_path}")

with open(gemini_api_key_path, "r", encoding="utf-8") as f:
    gemini_api_key = f.read().strip()

#initialize the Gemini API client with the API key
client = genai.Client(api_key=gemini_api_key)

In [ ]:
#calling the Gemini API to generate content based on the prompt template and raw notes

#replace placeholders in the prompt template with actual values
prompt = (
    prompt_template
    .replace("{today_date}", today)
    .replace("{timezone}", "Asia/Kolkata")
    .replace("{raw_notes}", raw_notes)
)

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt,
)

print(response.text)

[
  {
    "title": "ACE promotional layout meet",
    "event_type": "fixed_time",
    "anchor_datetime": "2026-08-16T18:00",
    "estimated_duration": null,
    "lock_status": "locked"
  },
  {
    "title": "ML pipeline debug - YOLO",
    "event_type": "fixed_time",
    "anchor_datetime": "2026-08-17T20:00",
    "estimated_duration": null,
    "lock_status": "movable"
  },
  {
    "title": "Calisthenics routine",
    "event_type": "fixed_time",
    "anchor_datetime": "2026-07-19T00:00",
    "estimated_duration": null,
    "lock_status": "movable"
  },
  {
    "title": "Shoot YouTube banter vid",
    "event_type": "deadline_task",
    "anchor_datetime": "2026-07-18T23:59",
    "estimated_duration": null,
    "lock_status": "movable"
  },
  {
    "title": "Ask papa about schedule",
    "event_type": "fixed_time",
    "anchor_datetime": "2026-07-17T00:00",
    "estimated_duration": null,
    "lock_status": "movable"
  },
  {
    "title": "Flute practice",
    "event_type": "deadline_task"

In [22]:
#storing the response from the Gemini API in a JSON file

response_text = response.text.strip()

# If Gemini returns pure JSON text
data = json.loads(response_text)

output_path = Path("data/export/response.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Saved response to {output_path}")

Saved response to data\export\response.json


In [26]:
with open("data/export/response.json", "r", encoding="utf-8") as f:
    tasks = json.load(f)

df = pd.DataFrame(tasks)

schema_columns = [
    "title",
    "event_type",
    "anchor_datetime",
    "estimated_duration",
    "lock_status",
]

df = df.reindex(columns=schema_columns)

df

,title,event_type,anchor_datetime,estimated_duration,lock_status
0,ACE promotional layout meet,fixed_time,2026-08-16T18:00,NaN,locked
1,ML pipeline debug - YOLO,fixed_time,2026-08-17T20:00,NaN,movable
2,Calisthenics routine,fixed_time,2026-07-19T00:00,NaN,movable
3,Shoot YouTube banter vid,deadline_task,2026-07-18T23:59,NaN,movable
4,Ask papa about schedule,fixed_time,2026-07-17T00:00,NaN,movable
5,Flute practice,deadline_task,NaN,30.0,movable
6,Finish CCUS report,deadline_task,2026-07-18T23:59,NaN,locked


In [31]:
# Open D-Tale editor
d = dtale.show(
    df,
    name="Scheduler task review",
    allow_cell_edits=True,
)

d.open_browser()

In [32]:
edited_df = d.data.copy()

edited_df = edited_df.reindex(columns=schema_columns)
edited_df = edited_df.astype(object)

edited_df.head()

,title,event_type,anchor_datetime,estimated_duration,lock_status
0,ACE promotional layout meet,fixed_time,2026-08-16T18:00,NaN,locked
1,ML pipeline debug - YOLO,fixed_time,2026-08-17T20:00,NaN,movable
2,Calisthenics routine,fixed_time,2026-07-19T00:00,NaN,movable
3,Shoot YouTube banter vid,deadline_task,2026-07-18T23:59,NaN,movable
4,Ask papa about schedule,fixed_time,2026-07-17T00:00,NaN,movable


In [34]:
edited_df = d.data.copy()

schema_columns = [
    "title",
    "event_type",
    "anchor_datetime",
    "estimated_duration",
    "lock_status",
]

edited_df = edited_df.reindex(columns=schema_columns)

edited_df.head()

# Clean text-ish columns
for col in ["title", "event_type", "anchor_datetime", "lock_status"]:
    edited_df[col] = edited_df[col].apply(
        lambda x: None if pd.isna(x) or str(x).strip() == "" else str(x).strip()
    )

# Clean duration column
edited_df["estimated_duration"] = edited_df["estimated_duration"].apply(
    lambda x: None if pd.isna(x) or str(x).strip() == "" else int(float(x))
)

edited_tasks = edited_df.to_dict(orient="records")

In [37]:
edited_df.head(10)

,title,event_type,anchor_datetime,estimated_duration,lock_status
0,ACE promotional layout meet,fixed_time,2026-08-16T18:00,NaN,locked
1,ML pipeline debug - YOLO,fixed_time,2026-08-17T20:00,NaN,movable
2,Calisthenics routine,fixed_time,2026-07-19T00:00,30.0,movable
3,Shoot YouTube banter vid,deadline_task,2026-07-18T23:59,NaN,movable
4,Ask papa about schedule,fixed_time,2026-07-17T00:00,NaN,movable
5,Flute practice,deadline_task,NaN,30.0,movable
6,Finish CCUS report,deadline_task,2026-07-18T23:59,NaN,locked


In [ ]:

tasks_json = edited_df.to_dict(orient="records")

with open("data/export/edited_response.json", "w", encoding="utf-8") as f:
    json.dump(tasks_json, f, indent=2, ensure_ascii=False)

print("Saved edited tasks to data/export/edited_response.json")

ValueError: cannot convert float NaN to integer